# Geração da Base de Colaboradores por Unidade/Dia

Gera a coluna `quantidade_colaboradores` (técnicos disponíveis) para cada empresa em cada dia.
- Varia de 5 a 12 técnicos
- Empresas com mais atendimentos em média recebem mais técnicos
- Há variação diária leve baseada no volume do dia vs. média da empresa

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('../Bases/VolumeDeAtendimentos.csv', sep=';')
df['data'] = pd.to_datetime(df['data'])
print(df.shape)
df.head()

In [ ]:
# Volume total por empresa por dia (soma de todos os produtos)
vol_dia = (
    df.groupby(['data', 'empresa'])['volume_atendimentos']
    .sum()
    .reset_index()
    .rename(columns={'volume_atendimentos': 'volume_total'})
)

# Media de volume por empresa (ao longo de todos os dias)
media_empresa = (
    vol_dia.groupby('empresa')['volume_total']
    .mean()
    .rename('media_volume')
)

print(f"Empresas: {media_empresa.shape[0]}")
print(f"Periodo: {vol_dia['data'].min().date()} a {vol_dia['data'].max().date()}")
media_empresa.describe()

In [ ]:
# Mapeamento: media de volume -> tecnicos base (5 a 12)
MIN_TECH = 5
MAX_TECH = 12

vol_min = media_empresa.min()
vol_max = media_empresa.max()

# Interpolacao linear: empresa com menor media -> 5 tecnicos; maior -> 12
techs_base = (
    (media_empresa - vol_min) / (vol_max - vol_min) * (MAX_TECH - MIN_TECH) + MIN_TECH
).round().astype(int).rename('techs_base')

techs_base = techs_base.clip(MIN_TECH, MAX_TECH)
print(techs_base.value_counts().sort_index())

In [ ]:
# Junta ao dataframe diario
vol_dia = vol_dia.join(media_empresa, on='empresa')
vol_dia = vol_dia.join(techs_base, on='empresa')

# Variacao diaria: +/-1 tecnico se volume do dia desvia >15% da media da empresa
ratio = vol_dia['volume_total'] / vol_dia['media_volume']

ajuste = np.where(ratio >= 1.15, 1, np.where(ratio <= 0.85, -1, 0))
vol_dia['quantidade_colaboradores'] = (vol_dia['techs_base'] + ajuste).clip(MIN_TECH, MAX_TECH)

print(vol_dia['quantidade_colaboradores'].value_counts().sort_index())
vol_dia.head(10)

In [ ]:
# Base final: data, empresa, quantidade_colaboradores
colaboradores = vol_dia[['data', 'empresa', 'quantidade_colaboradores']].copy()
colaboradores['data'] = colaboradores['data'].dt.strftime('%Y-%m-%d')

print(colaboradores.shape)
print(colaboradores['quantidade_colaboradores'].describe())
colaboradores.head(20)

In [ ]:
colaboradores.to_csv('../Bases/ColaboradoresPorDia.csv', sep=';', index=False)
print('Salvo em ../Bases/ColaboradoresPorDia.csv')